# Session 8 - attribution + encoder probe


**GPU T4 x2, ~2.5-3.5 h.** The mechanism analyses.

* **stated vs attributed vs causal** -- the region the model cites, where its gradients
  place mass on the visual tokens, and which region moves the verdict;
* **overlay-elicited citation** -- the citation prompt refers to a region outlined on
  the pixels;
* **encoder blind-spot probe** -- whether the visual tokens covering region k move
  under the counterfactual at all; if they do not, no language component could
  have cited that region faithfully.

**Cost note:** `--encoder-probe` runs two extra vision-tower passes per region, which
adds roughly 30-50% to the run time. The passes are cached, so the cost is paid once.

In [ ]:
SESSION = "S8 attribution"

# ============================== CONFIG ==============================
MODELS = ["qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct",
          "hfvlm:Qwen/Qwen2-VL-2B-Instruct"]
QUANT      = "fp16"
MAX_PIXELS = 384 * 384
SPLIT      = "test"
SEED       = 0
SAMPLES    = 600
CITE_MODE  = "both"      # text | overlay | both
ATTRIBUTION   = True
ENCODER_PROBE = True
BUDGET_PER_MODEL_MIN = 210

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils")
KU.gpu_report()
INDEX = KU.find_parsed_index()
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")
RESUME = ",".join(d for d in KU.find_run_dirs() if "run_" in d)
n_gpu = max(1, KU.n_gpus())

In [ ]:
# ==================== ATTRIBUTION + PROBE ====================
flags = f"--cite-mode {CITE_MODE}"
if ATTRIBUTION:
    flags += " --attribution"
if ENCODER_PROBE:
    flags += " --encoder-probe"
for model in MODELS:
    cmds, envs, logs = [], [], []
    for i in range(n_gpu):
        tag = "attr"
        logs.append(f"{OUT}/logs/attr_{C.safe_name(model)}_{i}.log")
        envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
        cmds.append(
            f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
            f'--detector "{model}" --out "{OUT}/run_attr" --tag {tag} '
            f'--split {SPLIT} --limit-samples {SAMPLES} --seed {SEED} '
            f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
            f'--max-pixels {MAX_PIXELS} {flags} --vocab {VOCAB} '
            f'--time-budget-min {BUDGET_PER_MODEL_MIN}'
            + (f' --resume-from "{RESUME}"' if RESUME else ""))
    print(C.banner(model))
    KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== ANALYSE ====================
KU.sh(f'{sys.executable} -m ccaudit.m14_attribution --raw "{OUT}/run_attr" '
      f'--out "{OUT}/attr" --vocab {VOCAB}', check=False)
blob = C.load_json(f"{OUT}/attr/attribution.json", {})
for e in blob.get("results", []):
    a, b = e.get("agreement", {}), e.get("blind_spot", {})
    print(f"\n{e['detector']}:")
    print(f"  stated = causal      {a.get('stated_vs_causal')}")
    print(f"  attributed = causal  {a.get('attr_vs_causal')}")
    print(f"  stated = attributed  {a.get('stated_vs_attr')}")
    if b:
        print(f"  encoder-blind        {b.get('encoder_blind_frac')}")
        print(f"  language confab      {b.get('language_confabulation_frac')}")
        print(f"  faithful             {b.get('faithful_frac')}")

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s8-attr`.
2. The three agreement rates (stated vs causal, attributed vs causal, stated vs
   attributed) are reported per model; they need not coincide, and the
   decomposition into encoder-blind and language-confabulation fractions
   locates the source of any disagreement.
3. If `attribute()` raised "could not locate the visual projector" for a model,
   that model has no gradient citation; report it as not supported rather
   than dropping the model."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)